# 02 - Data Cleaning and Quality Audit

**Purpose:** Parse timestamps, enforce regular frequency, document missing/sentinel values, duplicates, and outliers per dataset.

**Inputs:** data/raw/* (from notebook 01)

**Outputs:** Cleaned series under data/interim/, a per-dataset data-quality report.

**Runtime estimate:** 5-15 minutes.

## Timestamp parsing and frequency enforcement

In [1]:
from sdg_leakage_audit.data.loaders import DATASET_LOADERS, resolve_raw_file_path
from sdg_leakage_audit.data.manifest import load_manifest
from sdg_leakage_audit.utils.io import read_yaml
from sdg_leakage_audit.utils.paths import configs_dir, data_raw_dir, to_project_relative

datasets_cfg = read_yaml(configs_dir() / "datasets.yaml")["datasets"]
manifest = load_manifest()
raw_dir = data_raw_dir()

loaded = {}  # dataset_id -> parsed DataFrame (DatetimeIndex, structural parsing only)
for dataset_id, cfg in datasets_cfg.items():
    entry = manifest["datasets"].get(dataset_id)
    if entry is None:
        raise RuntimeError(f"No manifest entry for {dataset_id!r}; run notebook 01 first.")

    raw_path = resolve_raw_file_path(
        dataset_id=dataset_id,
        uci_id=cfg.get("uci_id"),
        resolved_route=entry["resolved_url"],
        raw_dir=raw_dir,
    )
    df = DATASET_LOADERS[dataset_id](raw_path)
    loaded[dataset_id] = df
    print(
        f"{dataset_id:32s} {to_project_relative(raw_path)!s:45s} "
        f"{len(df):>9,d} rows  [{df.index.min()} .. {df.index.max()}]"
    )


household_power_consumption      data\raw\uci_235_ucimlrepo.csv                2,075,259 rows  [2006-12-16 17:24:00 .. 2010-11-26 21:02:00]
appliances_energy_prediction     data\raw\uci_374_ucimlrepo.csv                   19,735 rows  [2016-01-11 17:00:00 .. 2016-05-27 18:00:00]
air_quality_uci                  data\raw\uci_360_ucimlrepo.csv                    9,357 rows  [2004-03-10 18:00:00 .. 2005-04-04 14:00:00]


2026-10-19 22:55:26 | INFO     | sdg_leakage_audit.data.loaders | Loaded Beijing multi-site air quality: 12 stations, 420768 rows total


beijing_multisite_air_quality    data\raw\uci_501.zip                            420,768 rows  [2013-03-01 00:00:00 .. 2017-02-28 23:00:00]


jena_climate                     data\raw\jena_climate.csv.zip                   420,551 rows  [2009-01-01 00:10:00 .. 2017-01-01 00:00:00]


## Missing and sentinel value handling (e.g. -200 in Air Quality)

In [2]:
from sdg_leakage_audit.data.cleaning import clean_and_report
from sdg_leakage_audit.utils.paths import data_interim_dir

data_interim_dir().mkdir(parents=True, exist_ok=True)

cleaned = {}  # dataset_id -> cleaned DataFrame (regular frequency, sentinel -> NaN)
quality_reports = {}  # dataset_id -> report dict

for dataset_id, cfg in datasets_cfg.items():
    df = loaded[dataset_id]
    cleaned_df, report = clean_and_report(
        df,
        freq=cfg["frequency"],
        target_column=cfg["target_column"],
        sentinel=cfg.get("sentinel_missing_value"),
        group_column="station" if dataset_id == "beijing_multisite_air_quality" else None,
    )
    cleaned[dataset_id] = cleaned_df
    quality_reports[dataset_id] = report

    out_path = data_interim_dir() / f"{dataset_id}.parquet"
    cleaned_df.to_parquet(out_path)

print("Cleaned and saved all datasets to data/interim/.")


2026-10-19 22:55:28 | INFO     | sdg_leakage_audit.data.cleaning | enforce_frequency: 2075259 -> 2075259 rows at freq=1min (25979 fully-missing gap rows introduced)


2026-10-19 22:55:29 | INFO     | sdg_leakage_audit.data.cleaning | enforce_frequency: 19735 -> 19735 rows at freq=10min (0 fully-missing gap rows introduced)


2026-10-19 22:55:29 | INFO     | sdg_leakage_audit.data.cleaning | replace_sentinel_values: replaced 1683 occurrences of -200 with NaN


2026-10-19 22:55:29 | INFO     | sdg_leakage_audit.data.cleaning | enforce_frequency: 9357 -> 9357 rows at freq=1h (0 fully-missing gap rows introduced)


2026-10-19 22:55:29 | INFO     | sdg_leakage_audit.data.cleaning | enforce_frequency: 35064 -> 35064 rows at freq=1h (0 fully-missing gap rows introduced)


...

2026-10-19 22:55:29 | INFO     | sdg_leakage_audit.data.cleaning | enforce_frequency: 35064 -> 35064 rows at freq=1h (0 fully-missing gap rows introduced)


2026-10-19 22:55:29 | INFO     | sdg_leakage_audit.data.cleaning | enforce_frequency: 35064 -> 35064 rows at freq=1h (0 fully-missing gap rows introduced)


2026-10-19 22:55:29 | WARNING  | sdg_leakage_audit.data.cleaning | enforce_frequency: 327 duplicate timestamps found; averaging duplicates before reindexing


2026-10-19 22:55:29 | INFO     | sdg_leakage_audit.data.cleaning | enforce_frequency: 420224 -> 420768 rows at freq=10min (544 fully-missing gap rows introduced)


Cleaned and saved all datasets to data/interim/.


## Duplicate detection

In [3]:
import pandas as pd

quality_df = pd.DataFrame.from_dict(quality_reports, orient="index")
quality_df.index.name = "dataset_id"
quality_df[["n_rows_raw", "n_duplicate_timestamps", "n_sentinel_replaced", "n_rows_after_freq", "n_gap_rows_introduced"]]


,n_rows_raw,n_duplicate_timestamps,n_sentinel_replaced,n_rows_after_freq,n_gap_rows_introduced
dataset_id,,,,,
household_power_consumption,2075259,0,0,2075259,0
appliances_energy_prediction,19735,0,0,19735,0
air_quality_uci,9357,0,1683,9357,0
beijing_multisite_air_quality,420768,0,0,420768,0
jena_climate,420551,327,0,420768,544


## Outlier detection

In [4]:
quality_df[["pct_missing_target_after_freq", "n_outliers_target"]]


,pct_missing_target_after_freq,n_outliers_target
dataset_id,,
household_power_consumption,1.251844,94907
appliances_energy_prediction,0.000000,2138
air_quality_uci,17.986534,215
beijing_multisite_air_quality,2.076916,19142
jena_climate,0.129287,1537


## Train-fit-only imputation note (leakage guard)

In [5]:
from sdg_leakage_audit.utils.paths import results_tables_dir, to_project_relative

out_path = results_tables_dir() / "data_quality_report.csv"
quality_df.to_csv(out_path)
print(f"Saved data quality report -> {to_project_relative(out_path)}")

print(
    "\nREMINDER (leakage guard): the sentinel replacement and frequency enforcement above "
    "are shared, deterministic data-hygiene steps applied identically regardless of any "
    "future train/test split -- they do not use any fitted statistic (mean, scaler, imputer) "
    "computed from the data. Any FITTED step (imputation, scaling, feature selection) must "
    "be fit on the train fold only; see protocols/injectors.py for the leakage-safe (P0) vs "
    "leaky (P2) comparison, and tests/test_no_peeking_guard.py for the automated check."
)


Saved data quality report -> results\tables\data_quality_report.csv

REMINDER (leakage guard): the sentinel replacement and frequency enforcement above are shared, deterministic data-hygiene steps applied identically regardless of any future train/test split -- they do not use any fitted statistic (mean, scaler, imputer) computed from the data. Any FITTED step (imputation, scaling, feature selection) must be fit on the train fold only; see protocols/injectors.py for the leakage-safe (P0) vs leaky (P2) comparison, and tests/test_no_peeking_guard.py for the automated check.


## Findings / what to carry forward

- All 5 datasets parsed to a standard `timestamp`-indexed DataFrame and cleaned (sentinel
  replacement, regular-frequency reindexing, duplicate/outlier flags); cleaned frames saved
  to `data/interim/<dataset_id>.parquet`. Full counts in `results/tables/data_quality_report.csv`.
- Household power consumption and Air Quality both use `"?"`/`-200` sentinels respectively --
  confirmed non-trivial missingness after frequency enforcement; feature engineering (notebook
  04) and modeling (06-07) must treat these as real gaps, not drop-and-forget.
- Beijing multi-site data required grouped-by-station frequency enforcement since each of the
  12 stations has its own series sharing an overlapping timestamp range; `clean_and_report`'s
  `group_column` parameter exists specifically for this case.
- Appliances Energy Prediction's raw `date` column has a missing-space formatting bug
  (`"2016-01-1117:00:00"`); fixed structurally in `loaders.load_appliances_energy_prediction`,
  not as a cleaning step, since it's a parsing issue rather than a data-quality issue.
- No fitted statistic (mean/scaler/imputer) was computed in this notebook -- cleaning here is
  deterministic and split-independent by construction, so it cannot leak test information
  into train. Actual imputation will be fit train-only inside the protocol pipeline (05+).
